# REGRESION LINEAL. REGRESIÓN LOGÍSTICA

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import warnings

warnings.filterwarnings('ignore')

In [ ]:
def categoricas_unicos(dataframe, var_cat, vc_out=True):
    """
    Muestra valores unicos y conteo de las variables categóricas.
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables categóricas (cadena)
    - vc_out: booleano para mostrar o no (True/False) el resultado de "value_counts()"
        Por defecto lo muestra

    -------------------------------
    SALIDA:
    No devuelve valor. Saca por pantalla la información

    """

    for discreta in var_cat:

        print(f'Variable {discreta.upper()}:')
        print('Valores unicos: ')
        print(dataframe[discreta].unique(), end='\n'*2)

        if vc_out:
            print(dataframe[discreta].value_counts(), end='\n'*2)





In [ ]:
def graficas_var_categorica(dataframe, var_cat):
    """
    Realiza los diagramas de barras de las variables categóricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """


    colores = sns.color_palette("husl", len(var_cat))

    # creacion matriz de graficas
    fig, axes = plt.subplots(len(var_cat), 1, \
                             figsize=(10, 4*len(var_cat)),\
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

    ax = axes.ravel()

    # dibujamos las graficas
    for idx,variable in enumerate(var_cat):

        sns.countplot(dataframe[variable], ax=ax[idx],palette=colores)

        ax[idx].set_title(f'DIAGRAMA DE BARRAS {variable}')
        ax[idx].set_xlabel(f'Valores únicos (categorias) de {variable}')
        ax[idx].set_ylabel("Frequencia")


In [ ]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.distplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     hist_kws={'alpha': 0.15})

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')



## CARGA DEL ARCHIVO

Vamos a utilizar un archivo de datos de una aseguradora para preveer los gastos que puede ocasionar un cliente en función de otras variables: "**insurance.csv"**. Lo cargamos y echamos un vistazo

In [ ]:
RUTA = './insurance.csv'
insurance_df = pd.read_csv(RUTA)

insurance_df

In [ ]:
insurance_df.columns

In [ ]:
insurance_df.info()

**Es mas comodo separar variables:**

In [ ]:
target = 'charges'

var_num = insurance_df.select_dtypes(exclude=['object']).columns.to_list()

var_cat = insurance_df.select_dtypes(include=['object']).columns.to_list()

var_num.remove(target)

var_num

Vemos la distribución de las categóricas y las numéricas:

In [ ]:
categoricas_unicos(insurance_df,var_cat)

In [ ]:
graficas_var_categorica(insurance_df, var_cat)


**Ahora las numericas:**

In [ ]:
insurance_df.describe()

In [ ]:
graf_histo_box_numericas(insurance_df, var_num)

Vemos que BMI presenta algunos, si bien no muy alejados y que no podemos descartar como no posibles. **Por interés didactico los dejamos como estan**

# Echemos un vistazo a la correlación

In [ ]:
mat_corr = insurance_df.corr()
mat_corr

In [ ]:
sns.heatmap(mat_corr, annot=True, cbar=True, \
            cmap=sns.color_palette("coolwarm", as_cmap=True))
plt.title('Correlación entre atributos');

**No parece haber dependencias.**

Separamos las distintas variables:

In [ ]:
X = insurance_df.drop(columns=target)

y = insurance_df[target]

X.shape, y.shape

## CODIFICACIÓN A NUMÉRICO: OHE (antes de la partición)

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse=False, dtype=np.integer, drop='first')

X_cat = ohe.fit_transform(X[var_cat])

Xcat_df = pd.DataFrame(data=X_cat, columns=ohe.get_feature_names_out())

Xcat_df

In [ ]:
X1 = pd.concat((X[var_num], Xcat_df), axis=1)

X1

**Hay que actualizar los nombres de las categóricas después del procesado:**

In [ ]:
var_cat = ohe.get_feature_names_out()

var_cat

## PARTICIÓN TRAIN TEST

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

### Modelo Base: kNN

Nos valemos de un modelo base para un primer tanteo:

In [ ]:
knn_regr = KNeighborsRegressor(n_neighbors=5)

knn_regr.fit(X_train, y_train)

y_pred = knn_regr.predict(X_test)

mean_squared_error(y_test, y_pred, squared=False)

## REGRESIÓN LINEAL

In [ ]:
linregr = LinearRegression()

linregr.fit(X_train, y_train)

y_pred_train = linregr.predict(X_train)
y_pred_test = linregr.predict(X_test)


In [ ]:
mean_squared_error(y_train, y_pred_train, squared=False)

In [ ]:
mean_squared_error(y_test, y_pred_test, squared=False)

Vemos con métrica R2:

In [ ]:
linregr.score(X_train, y_train)

In [ ]:
linregr.score(X_test, y_test)

#### Para utilizar los modelos regularizados necesitamos estandarizar:

In [ ]:
scaler = StandardScaler()

Xnum_train_scaled = scaler.fit_transform(X_train[var_num])

Xnum_test_scaled = scaler.transform(X_test[var_num])

df_Xnum_train_scaled = pd.DataFrame(data=Xnum_train_scaled, columns=scaler.get_feature_names_out(), index=X_train.index)

df_Xnum_test_scaled = pd.DataFrame(data=Xnum_test_scaled, columns=scaler.get_feature_names_out(), index=X_test.index)

df_Xnum_train_scaled



In [ ]:
df_Xnum_test_scaled

In [ ]:
Xtrain = pd.concat((df_Xnum_train_scaled, X_train[var_cat]), axis=1)

Xtest= pd.concat((df_Xnum_test_scaled, X_test[var_cat]), axis=1)

Xtrain.head()

In [ ]:
Xtrain.shape, Xtest.shape

In [ ]:
np.logspace(-1,2, num=4)

## RIDGE

In [ ]:
alfas = np.logspace(-1,2, num=4)

rmse_train = []
rmse_test = []

coefs = dict()

for alfa in alfas:

    ridge = Ridge(alpha=alfa)
    ridge.fit(Xtrain, y_train)

    y_pred_train = ridge.predict(Xtrain)
    y_pred_test = ridge.predict(Xtest)

    rmse_train.append(mean_squared_error(y_train, y_pred_train, squared=False))
    rmse_test.append(mean_squared_error(y_test, y_pred_test, squared=False))

    coefs.update({str(alfa): ridge.coef_})



In [ ]:
pd.DataFrame({'alpha': alfas, 'RMSE_TRAIN': rmse_train, 'RMSE_TEST': rmse_test})

### Interpretación de los coeficientes

In [ ]:
for alfa, coefic in coefs.items():

    fig, ax = plt.subplots(figsize=(8, 3))

    sns.barplot(x=Xtrain.columns, y=coefic, ax=ax)

    plt.xticks(rotation=90, ha='right', size=10)
    ax.set_xlabel('atributos')
    ax.set_ylabel('coeficientes')
    ax.set_title(f'Coeficientes RIDGE: alfa={alfa}');





In [ ]:
for alfa, coefic in coefs.items():

    print(f'Valor de alfa: {alfa}')
    print(coefic)
    print('\n'*2)

In [ ]:
Xtrain.columns

## LASSO

In [ ]:
alfas = np.logspace(-1,2, num=4)

rmse_train = []
rmse_test = []

coefs = dict()

for alfa in alfas:

    lasso = Lasso(alpha=alfa)
    lasso.fit(Xtrain, y_train)

    y_pred_train = lasso.predict(Xtrain)
    y_pred_test = lasso.predict(Xtest)

    rmse_train.append(mean_squared_error(y_train, y_pred_train, squared=False))
    rmse_test.append(mean_squared_error(y_test, y_pred_test, squared=False))

    coefs.update({str(alfa): lasso.coef_})



In [ ]:
pd.DataFrame({'alpha': alfas, 'RMSE_TRAIN': rmse_train, 'RMSE_TEST': rmse_test})

In [ ]:
for alfa, coefic in coefs.items():

    fig, ax = plt.subplots(figsize=(8, 3))

    sns.barplot(x=Xtrain.columns, y=coefic, ax=ax)

    plt.xticks(rotation=90, ha='right', size=10)
    ax.set_xlabel('atributos')
    ax.set_ylabel('coeficientes')
    ax.set_title(f'Coeficientes LASSO: alfa={alfa}');





In [ ]:
for alfa, coefic in coefs.items():

    print(coefic)

Vemos que si aumentamos mucho la regularización, algunos coeficientes se anulan **(LASSO "realiza" selección de atributos)**


# REGRESIÓN LOGÍSTICA

Probamos la regresión logística con el dataset iris, para ver las fronteras de decisión:

In [ ]:
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True, as_frame=True)

X.shape, y.shape

In [ ]:
X.head()

In [ ]:
# "arreglamos" nombres columnas atributos
columnas = [columna[:-5].replace(" ","_") for columna in X.columns]

X.columns = columnas

X.columns

In [ ]:
y.unique()

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [ ]:
y.value_counts()

Realizamos el particionado:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Ahora si podemos escalar

In [ ]:
scaler = StandardScaler()

Xtrain_scaled = scaler.fit_transform(X_train)

Xtest_scaled = scaler.transform(X_test)

df_Xtrain_scaled = pd.DataFrame(data=Xtrain_scaled, columns=scaler.get_feature_names_out(), index=X_train.index)

df_Xtest_scaled = pd.DataFrame(data=Xtest_scaled, columns=scaler.get_feature_names_out(), index=X_test.index)

df_Xtrain_scaled



Probamos con varios valores de "C". Cuando éste aumenta, la regularización disminuye produciendo modelos mas complejos.

In [ ]:
C_valores = np.logspace(-3,2, num=6)

acc_train = []
acc_test = []


for c in C_valores:

    logist = LogisticRegression(C=c)
    logist.fit(Xtrain_scaled, y_train)


    acc_train.append(logist.score(Xtrain_scaled, y_train))
    acc_test.append(logist.score(Xtest_scaled, y_test))




In [ ]:
pd.DataFrame({'C': C_valores, 'acc_train': acc_train, 'acc_test': acc_test})

Comprobamos que **con el valor mas alto de "C" (nada de regularización** conseguimos el máximo de "accuracy" (% acierto) para train, señalando sobreentrenamiento.

Según bajamos "C" aumenta la regularización obteniendo modelos más sencillos.

## Comparación de modelos según regularización L2 "C". Fronteras de decisión

Volvemos a utilizar la **función** que dibuja las fronteras de decisión:

In [ ]:
from matplotlib.colors import ListedColormap


# Definimos la función que nos graficará las fronteras de decisión

def plot_decision_boundaries(model, X, y, delta: float = .02) -> None:
    """Plot data points and deicision boundaries learned by the model.

    Arguments:
    ----------
    model: scikit-learn like model

    X: np.array[n_samples, n_features]
        Only first 2 features will be considered because it is a 2d plot.
        Feature 0 in the x axis, and feature 1 in the y axis.

    y: np.array
        Labels for each sample.

    delta: float
        Increment between consecutive points when computing the grid for plotting boundaries.
        Lower value for higher resolution.
    """

    # Creamos la meshgrid con los valores mínimo y máximo de 'x' i 'y'.
    # La variable X es nuestro dataframe con las variables a estudiar (las del pétalo o las del sépalo)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, delta),
                         np.arange(y_min, y_max, delta))

    #Predecimos el clasificador con los valores de la meshgrid
    # En este caso model será nuestra variable que contiene el modelo a estudiar, es decir K-nn, SVM,...
    # Por ejemplo para K-nn sería model = KNeighborsClassifier()

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])

    # Creamos mapas de colores con ListedColormap para ver como separa las clases.
    # En este caso usaremos:
    # Iris-setosa : darkorange
    # Iris-versicolor: c
    # Iris-virginica: darkblue

    cmap_light = ListedColormap(['orange', 'cyan', 'cornflowerblue'])
    cmap_bold = ListedColormap(['darkorange', 'c', 'darkblue'])

    # Ponemos el resultado en una figura de color
    Z = Z.reshape(xx.shape)
    plt.figure()
    plt.pcolormesh(xx, yy, Z, cmap= cmap_light)

    # Dibujamos también los puntos de entrenamiento
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap= cmap_bold)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.show()

Para poder visualizarla en una gráfica 2D, elegiremos sólo 2 de los atributos. Si recuerdan los mejores atributos que "separaban" las clases eran las del pétalo ("petal_length", "petal_width"):

In [ ]:
X_train_f, X_test_f = Xtrain_scaled[:,2:], Xtest_scaled[:,2:]

In [ ]:
C_valores = np.logspace(-3,2, num=6)

modelos_f = []

for c in C_valores:

    logist = LogisticRegression(C=c)
    logist.fit(X_train_f, y_train)

    modelos_f.append(logist)



In [ ]:
modelos_f[0]

Veamos las gráficas:

## C = 0.001

In [ ]:
plot_decision_boundaries(modelos_f[0], X_train_f, y_train)

\

## C = 0.01

In [ ]:
plot_decision_boundaries(modelos_f[1], X_train_f, y_train)

## C = 0.1

In [ ]:
plot_decision_boundaries(modelos_f[2], X_train_f, y_train)

![download.png](attachment:e3664048-6d30-414b-8b22-db2d776a7d92.png)## C = 1

In [ ]:
plot_decision_boundaries(modelos_f[3], X_train_f, y_train)

## C = 10

In [ ]:
plot_decision_boundaries(modelos_f[4], X_train_f, y_train)

## C = 100

In [ ]:
plot_decision_boundaries(modelos_f[5], X_train_f, y_train)

La **evolución según los valores de "C"** empieza con valor muy bajo de C, que implica muy regularizado, tanto que entra en underfitting (primera gráfica). Segun crece "C" disminuye la regularización y vemos que las fronteras de decisión se ajustan más a los datos TRAIN.

## EJERCICIO DIABETES

#### Carga del dataset:

In [ ]:
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True, as_frame=True)

X.shape, y.shape

In [ ]:
X.info()

**Todos los atributos numéricos, incluido "sexo" (ya codificado a numérico)**

In [ ]:
X.sex.value_counts()

In [ ]:
X.describe()

**Se puede observar que ya los atributos están escalados**

In [ ]:
sns.distplot(y)

**Target con ligero sesgo o cola a la derecha a la derecha, seguimos adelante con las regresiones**

**Veamos la correlación:**

In [ ]:
mat_corr = X.corr()

mat_corr

In [ ]:
sns.heatmap(mat_corr, annot=True, cbar=True, cmap=sns.color_palette("coolwarm", as_cmap=True));

Observamos varios atributos correlados. Decidimos las columnas a eliminar:

eliminamos "s2", "s4"

In [ ]:
X_corr = X.drop(columns=['s2','s4'])

X_corr.columns

#### Partición TRAIN/TEST

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_corr, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

**Modelo Base: kNN**

In [ ]:
knn_regr = KNeighborsRegressor(n_neighbors=5)

knn_regr.fit(X_train, y_train)

y_pred = knn_regr.predict(X_test)

mean_squared_error(y_test, y_pred, squared=False)

In [ ]:
knn_regr.score(X_train, y_train)

In [ ]:
knn_regr.score(X_test, y_test)

#### REGRESIÓN LINEAL

In [ ]:
linregr = LinearRegression()

linregr.fit(X_train, y_train)

y_pred_train = linregr.predict(X_train)
y_pred_test = linregr.predict(X_test)


In [ ]:
mean_squared_error(y_train, y_pred_train, squared=False)

In [ ]:
mean_squared_error(y_test, y_pred_test, squared=False)

Vemos con métrica R2:

In [ ]:
linregr.score(X_train, y_train)

In [ ]:
linregr.score(X_test, y_test)

**No conseguimos mejora**. Nos vamos a los modelos regularizados

#### RIDGE con al menos 2 valores de "alfa"

In [ ]:
alfas = np.logspace(-1,2, num=4)

rmse_train = []
rmse_test = []

coefs = dict()

modelos_alfa = []

for alfa in alfas:

    ridge = Ridge(alpha=alfa)
    ridge.fit(X_train, y_train)

    y_pred_train = ridge.predict(X_train)
    y_pred_test = ridge.predict(X_test)

    rmse_train.append(mean_squared_error(y_train, y_pred_train, squared=False))
    rmse_test.append(mean_squared_error(y_test, y_pred_test, squared=False))

    modelos_alfa.append(ridge)

    coefs.update({str(alfa): ridge.coef_})



In [ ]:
pd.DataFrame({'alpha': alfas, 'RMSE_TRAIN': rmse_train, 'RMSE_TEST': rmse_test})

In [ ]:
modelos_alfa[0]

In [ ]:
modelos_alfa[0].score(X_train, y_train)

In [ ]:
modelos_alfa[0].score(X_test, y_test)

**Interpretación de los coeficientes**

In [ ]:
for alfa, coefic in coefs.items():

    fig, ax = plt.subplots(figsize=(8, 3))

    sns.barplot(x=X_corr.columns, y=coefic, ax=ax)

    plt.xticks(rotation=90, ha='right', size=10)
    ax.set_xlabel('atributos')
    ax.set_ylabel('coeficientes')
    ax.set_title(f'Coeficientes RIDGE: alfa={alfa}');





**NOTA IMPORTANTE:** Al estar los datos escalados estudiar los coeficientes en magnitud es mucho mas complicado (mas si como es el caso no conocemos cómo se ha escalado), se suele revisar solo lo anterior, la tendencia

#### LASSO con al menos 2 valores de "alfa"

In [ ]:
alfas = np.logspace(-1,2, num=4)

rmse_train = []
rmse_test = []

coefs = dict()

modelos_alfa = []

for alfa in alfas:

    lasso = Lasso(alpha=alfa)
    lasso.fit(X_train, y_train)

    y_pred_train = lasso.predict(X_train)
    y_pred_test = lasso.predict(X_test)

    rmse_train.append(mean_squared_error(y_train, y_pred_train, squared=False))
    rmse_test.append(mean_squared_error(y_test, y_pred_test, squared=False))

    modelos_alfa.append(lasso)

    coefs.update({str(alfa): lasso.coef_})



In [ ]:
pd.DataFrame({'alpha': alfas, 'RMSE_TRAIN': rmse_train, 'RMSE_TEST': rmse_test})

In [ ]:
modelos_alfa[0]

In [ ]:
modelos_alfa[0].score(X_train, y_train)

In [ ]:
modelos_alfa[0].score(X_test, y_test)

**Interpretación de los coeficientes**

In [ ]:
for alfa, coefic in coefs.items():

    fig, ax = plt.subplots(figsize=(8, 3))

    sns.barplot(x=X_corr.columns, y=coefic, ax=ax)

    plt.xticks(rotation=90, ha='right', size=10)
    ax.set_xlabel('atributos')
    ax.set_ylabel('coeficientes')
    ax.set_title(f'Coeficientes LASSO: alfa={alfa}');





#### EXTRA: transformaciones matematicas en busca de "normalidad" target/atributos

Vamos a transformar el target para conseguir disminuir la cola, hacer que se parezca algo más a una distribución normal:

In [ ]:
y_trans = np.log1p(y)

fig, ax = plt.subplots(1,2, figsize=(10,4))

sns.distplot(y, bins=30, ax=ax[0])

#ax[0].set_xlim([0, 2000])
ax[0].set_ylabel('Frecuencia')
ax[0].set_xlabel('Target')
ax[0].set_title('Target original')

sns.distplot(y_trans, bins=30, ax=ax[1])

#ax[1].set_xlim([0, 2000])
ax[1].set_ylabel('Frecuencia')
ax[1].set_xlabel('Target')
ax[1].set_title('Target TRANSFORMADO');

In [ ]:
y_trans2 = np.sqrt(y)

fig, ax = plt.subplots(1,2, figsize=(10,4))

sns.distplot(y, bins=30, ax=ax[0])

ax[0].set_ylabel('Frecuencia')
ax[0].set_xlabel('Target')
ax[0].set_title('Target original')

sns.distplot(y_trans2, bins=30, ax=ax[1])

ax[1].set_ylabel('Frecuencia')
ax[1].set_xlabel('Target')
ax[1].set_title('Target TRANSFORMADO');

- Para cambiar las variables también se utilizan transformadores, uno muy utilixzado es **"PowerTransformer"** que realiza las transformaciones Box-Cox (variable estrictamente positiva) y Yeo-Jhonson. Otro transformador es el **"QuantileTransformer"**, al que hay que indicarle con un parámetro que se requiere una normal.

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='yeo-johnson', standardize=True)

y_trans3 = pt.fit_transform(y.values.reshape(-1,1))

fig, ax = plt.subplots(1,2, figsize=(10,4))

sns.distplot(y, bins=30, ax=ax[0])

ax[0].set_ylabel('Frecuencia')
ax[0].set_xlabel('Target')
ax[0].set_title('Target original')

sns.distplot(y_trans3, bins=30, ax=ax[1])

ax[1].set_ylabel('Frecuencia')
ax[1].set_xlabel('Target')
ax[1].set_title('Target TRANSFORMADO');

#### Transformador "TransformedTargetRegressor"

**Scikit-Learn ofrece un modelo que facilita mucho la tarea de trabajar con target transformado**, es el **"TransformedTargetRegressor"**. Tiene en cuenta las operaciones/transformadores utilizados para darnos la respuesta adecuada, realizando el modelo todos los cálculos internos necesarios.

Vamos a probar con la última operación (sqrt(y)) y con el transformador "PowerTransformer":

- **FUNCION SQRT(y)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_corr, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.compose import TransformedTargetRegressor

In [ ]:
regr_trans_f = TransformedTargetRegressor(regressor=Ridge(alpha=0.1), func=np.sqrt, inverse_func=np.square)

regr_trans_f.fit(X_train, y_train)

y_pred_train_f = regr_trans_f.predict(X_train)
y_pred_test_f = regr_trans_f.predict(X_test)

**Veamos los errores RMSE:**

In [ ]:
mean_squared_error(y_train, y_pred_train_f, squared=False)

In [ ]:
mean_squared_error(y_test, y_pred_test_f, squared=False)

Vemos con métrica R2:

In [ ]:
regr_trans_f.score(X_train, y_train)

In [ ]:
regr_trans_f.score(X_test, y_test)

**Muy ligera mejoría, prácticamente inapreciable**

- **TRANSFORMADOR "PowerTransformer":**

In [ ]:
regr_trans_t = TransformedTargetRegressor(regressor=Ridge(alpha=0.1), \
                                          transformer=PowerTransformer(method='yeo-johnson', standardize=True))

regr_trans_t.fit(X_train, y_train)

y_pred_train_t = regr_trans_t.predict(X_train)
y_pred_test_t = regr_trans_t.predict(X_test)

**Veamos los errores RMSE:**

In [ ]:
mean_squared_error(y_train, y_pred_train_t, squared=False)

In [ ]:
mean_squared_error(y_test, y_pred_test_t, squared=False)

Vemos con métrica R2:

In [ ]:
regr_trans_t.score(X_train, y_train)

In [ ]:
regr_trans_t.score(X_test, y_test)

**No se observa mejora, en este caso con toda seguridad el tamaño y dimensionalidad del dataset sigue siendo un "lastre"**

In [ ]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)

type(X), X.shape, type(y), y.shape

In [ ]:
X.info()

In [ ]:
sns.distplot(y)

In [ ]:
y.describe()

In [ ]:
df_house = pd.concat([X,y], axis=1)

df_house

In [ ]:
sns.pairplot(df_house, diag_kind='kde');

In [ ]:
mat_corr = df_house.corr()

mat_corr

In [ ]:
sns.heatmap(mat_corr, annot=True, cbar=True, \
            cmap=sns.color_palette("coolwarm", as_cmap=True));

In [ ]:
df_house.columns

In [ ]:
df_house_regr = df_house.drop(columns=['AveBedrms', 'Longitude'])

df_house_regr.columns

In [ ]:
target = 'MedHouseVal'

X = df_house_regr.drop(columns=target)

y = df_house_regr[target]

X.shape, y.shape

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

In [ ]:
std_scaler = StandardScaler()

Xtrain_esc = std_scaler.fit_transform(Xtrain)

Xtest_esc = std_scaler.transform(Xtest)



In [ ]:

lr = LinearRegression()
lr.fit(Xtrain_esc, ytrain)

y_pred_train = lr.predict(Xtrain_esc)
y_pred_test = lr.predict(Xtest_esc)

rmse_train = mean_squared_error(ytrain, y_pred_train, squared=False)
rmse_test = mean_squared_error(ytest, y_pred_test, squared=False)

r2_train = lr.score(Xtrain_esc, ytrain)
r2_test = lr.score(Xtest_esc, ytest)

coefs = lr.coef_

print(f'TRAIN: rmse {rmse_train}, r2 {r2_train}')

print(f'TEST: rmse {rmse_test}, r2 {r2_test}')



In [ ]:
Q1, Q3 = y.quantile([0.25,0.75])

IQR = Q3-Q1

lim_sup = Q3 + 1.5*IQR

lim_sup

In [ ]:
idx_borrar = y[y>lim_sup]

len(idx_borrar)

In [ ]:
X